# Task 14: TMD/靶向特征 — in_TMD + dist_to_nearest_TMD + delta_membrane_insertion

**目的**: 突变是否落在跨膜段 (Transmembrane Domain) 内、离 TMD 多远、以及对膜插入自由能的影响，这三个特征携带与 `delta_hydrophobicity` 正交的结构信息。

| 新特征 | 类型 | 说明 |
|---|---|---|
| `in_TMD` | 二值 | 突变位点是否在跨膜段内 |
| `dist_to_nearest_TMD` | [0,1] 连续 | 到最近 TMD 边界的归一化距离 (1=段内, →0=无限远) |
| `delta_membrane_insertion` | 连续 | 突变前后 TMD 段中心加权膜插入 ΔG 变化 (Hessa 尺度) |

**输出**: `tmd_features.csv` (KEY + 3 列)，下游任务按 `Gene||Mutation_used` 对齐加载。

In [1]:
import re, json, time, warnings
import functools
import numpy as np
import pandas as pd
import requests
warnings.filterwarnings("ignore")

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"
SRC_PATH  = "/mnt/volume6/czj/labLGN/LabLZ/data_preparation/cell2024_model_single_subst.csv"

# ===== Hessa 生物学膜插入 ΔG 尺度 (kcal/mol, 越负越利于插入) =====
# 来源: Hessa et al. 2005, Nature; 常用近似值
HESSA = {
    'A': 0.11, 'R': 2.58, 'N': 2.05, 'D': 3.49, 'C': -0.13,
    'Q': 2.36, 'E': 2.68, 'G': 0.74, 'H': 2.06, 'I': -0.60,
    'L': -0.55, 'K': 2.71, 'M': -0.10, 'F': -0.32, 'P': 2.23,
    'S': 0.84, 'T': 0.52, 'W': -0.27, 'Y': -0.31, 'V': -0.31
}

# Kyte-Doolittle 疏水尺度 (备用，本次不使用，仅保留以供对比)
KD = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}


def parse_mutation(s):
    """解析突变字符串: "K255E" 或 "ACTN4 K255E" → (wt, pos, mt)"""
    if not isinstance(s, str):
        return None, None, None
    m = re.search(r'([A-Z])(\d+)([A-Z])', s)
    if m is None:
        return None, None, None
    return m.group(1), int(m.group(2)), m.group(3)

print("工具函数就绪.")

工具函数就绪.


## 14a. 拉 UniProt 跨膜段 + 序列

对每个唯一的 UniProt accession，通过 REST API 获取 canonical 序列和 Transmembrane 注释。
用 `lru_cache` 缓存避免重复请求。

In [2]:
@functools.lru_cache(maxsize=None)
def fetch_uniprot(acc):
    """获取 UniProt accession 的 canonical 序列和跨膜段列表。
    
    Returns:
        seq: str — canonical 序列
        tmds: list of (start, end) — 跨膜段 1-based 起止坐标
    """
    url = f"https://rest.uniprot.org/uniprotkb/{acc}.json"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    d = r.json()
    
    seq = d["sequence"]["value"]
    
    tmds = []
    for f in d.get("features", []):
        if f.get("type") == "Transmembrane":
            s = f["location"]["start"]["value"]
            e = f["location"]["end"]["value"]
            if s and e:  # 跳过不确定位点
                tmds.append((int(s), int(e)))
    
    return seq, tmds


# ===== 预加载所有唯一 UniProt =====
df_src = pd.read_csv(SRC_PATH)
unique_accs = df_src["Uniprot"].dropna().unique().tolist()
print(f"唯一 UniProt accession: {len(unique_accs)}")

# 拉取所有 (带进度和错误处理)
uniprot_cache = {}      # acc → (seq, tmds)
fetch_errors = {}       # acc → error_msg

t0 = time.time()
for i, acc in enumerate(unique_accs):
    try:
        uniprot_cache[acc] = fetch_uniprot(acc)
    except Exception as e:
        fetch_errors[acc] = str(e)
        uniprot_cache[acc] = (None, [])  # 无注释时序列置 None, TMD 置空
    
    if (i + 1) % 50 == 0 or i == 0 or i == len(unique_accs) - 1:
        elapsed = time.time() - t0
        n_ok = len(unique_accs) - len(fetch_errors)
        n_tmd = sum(1 for v in uniprot_cache.values() if v[1])
        print(f"  {i+1}/{len(unique_accs)}: {n_ok} ok, {n_tmd} 有 TMD 注释, "
              f"{len(fetch_errors)} 失败, 耗时={elapsed:.0f}s")

print(f"\n拉取完成. 成功={len(unique_accs)-len(fetch_errors)}, "
      f"有TMD注释={sum(1 for v in uniprot_cache.values() if v[1])}, "
      f"失败={len(fetch_errors)}")
if fetch_errors:
    print(f"失败的 accession: {list(fetch_errors.keys())[:10]}...")

唯一 UniProt accession: 871
  1/871: 871 ok, 0 有 TMD 注释, 0 失败, 耗时=2s
  50/871: 871 ok, 10 有 TMD 注释, 0 失败, 耗时=74s
  100/871: 871 ok, 21 有 TMD 注释, 0 失败, 耗时=144s
  150/871: 871 ok, 35 有 TMD 注释, 0 失败, 耗时=215s
  200/871: 871 ok, 56 有 TMD 注释, 0 失败, 耗时=282s
  250/871: 871 ok, 67 有 TMD 注释, 0 失败, 耗时=357s
  300/871: 871 ok, 84 有 TMD 注释, 0 失败, 耗时=428s
  350/871: 871 ok, 102 有 TMD 注释, 0 失败, 耗时=499s
  400/871: 871 ok, 113 有 TMD 注释, 0 失败, 耗时=574s
  450/871: 871 ok, 133 有 TMD 注释, 0 失败, 耗时=646s
  500/871: 871 ok, 143 有 TMD 注释, 0 失败, 耗时=715s
  550/871: 871 ok, 155 有 TMD 注释, 0 失败, 耗时=784s
  600/871: 871 ok, 164 有 TMD 注释, 0 失败, 耗时=852s
  650/871: 871 ok, 188 有 TMD 注释, 0 失败, 耗时=920s
  700/871: 871 ok, 212 有 TMD 注释, 0 失败, 耗时=989s
  750/871: 871 ok, 223 有 TMD 注释, 0 失败, 耗时=1059s
  800/871: 871 ok, 242 有 TMD 注释, 0 失败, 耗时=1128s
  850/871: 871 ok, 254 有 TMD 注释, 0 失败, 耗时=1199s
  871/871: 871 ok, 259 有 TMD 注释, 0 失败, 耗时=1229s

拉取完成. 成功=871, 有TMD注释=259, 失败=0


## 14b. 特征计算函数

In [3]:
# ---- in_TMD & dist_to_nearest_TMD ----

def tmd_hit_and_dist(pos, tmds):
    """判断位点是否在 TMD 内，以及到最近 TMD 的归一化距离。
    
    Args:
        pos: int, 1-based 突变位置
        tmds: list of (start, end)
    
    Returns:
        in_tmd: 0 或 1
        prox: float, 1/(1+d) — 段内=1, 越远→0
    """
    if not tmds:
        return 0, 0.0  # 无注释: in_TMD=0, 距离=0 (等价无限远)
    
    in_tmd = int(any(s <= pos <= e for s, e in tmds))
    
    if in_tmd:
        return 1, 1.0
    
    d = min(min(abs(pos - s), abs(pos - e)) for s, e in tmds)
    prox = 1.0 / (1.0 + d)  # 归一化: 段内=1, 紧邻≈0.5, 远处→0
    return 0, prox


# ---- delta_membrane_insertion ----

def segment_insertion_dG(seq, seg_start, seg_end, scale):
    """计算一段序列的中心加权膜插入 ΔG。
    
    关键: 窗口固定在整段 TMD，不是以突变为中心;
    中心加权使螺旋中段贡献更大 → 不与逐残基 delta_hydrophobicity 等价。
    """
    sub = seq[seg_start - 1:seg_end]
    L = len(sub)
    if L == 0:
        return np.nan
    
    idx = np.arange(L)
    center = (L - 1) / 2
    sigma = L / 4.0
    w = np.exp(-((idx - center) ** 2) / (2 * sigma ** 2))
    w /= w.sum()
    
    vals = np.array([scale.get(aa, 0.0) for aa in sub])
    return float((w * vals).sum())


def containing_segment(pos, tmds):
    """返回包含 pos 的 TMD 段 (start, end)，若不在任何段内则返回 None。"""
    for s, e in tmds:
        if s <= pos <= e:
            return (s, e)
    return None


def delta_membrane_insertion(wt_seq, pos, wt, mt, tmds, scale=HESSA):
    """突变前后 TMD 段膜插入 ΔG 变化。
    
    Returns:
        float: ΔΔG_membrane_insertion, 
               0.0 如果不在任何 TMD 内 (机制不适用),
               np.nan 如果 canonical 残基与 WT 不符 (数据问题)
    """
    seg = containing_segment(pos, tmds)
    if seg is None:
        return 0.0
    
    # 校验: canonical 序列上的残基应与 WT 一致
    if pos < 1 or pos > len(wt_seq):
        return np.nan
    if wt_seq[pos - 1] != wt:
        return np.nan
    
    mt_seq = wt_seq[:pos - 1] + mt + wt_seq[pos:]
    dG_wt = segment_insertion_dG(wt_seq, *seg, scale)
    dG_mt = segment_insertion_dG(mt_seq, *seg, scale)
    return dG_mt - dG_wt

print("特征计算函数就绪.")

特征计算函数就绪.


## 14c. 批量计算 + 组装

In [4]:
rows = []
wt_mismatch_log = []  # 记录 canonical 残基与 WT 不符的条目

for _, r in df_src.iterrows():
    acc = r["Uniprot"]
    key = r["Gene"] + "||" + r["Mutation_used"]
    
    # 解析突变
    wt, pos, mt = parse_mutation(r["Mutation_used"])
    if wt is None:
        rows.append({"KEY": key, "in_TMD": 0,
                     "dist_to_nearest_TMD": 0.0,
                     "delta_membrane_insertion": 0.0})
        continue
    
    # 取 UniProt 缓存
    seq, tmds = uniprot_cache.get(acc, (None, []))
    
    # in_TMD & dist
    in_tmd, prox = tmd_hit_and_dist(pos, tmds)
    
    # delta_membrane_insertion
    if seq is None or not tmds:
        dmi = 0.0  # 无序列或无用 TMD → 机制不适用
    else:
        dmi = delta_membrane_insertion(seq, pos, wt, mt, tmds)
        if np.isnan(dmi):
            wt_mismatch_log.append((key, acc, wt, r["Mutation_used"], pos))
            dmi = 0.0  # 不一致时保守置 0
    
    rows.append({"KEY": key, "in_TMD": in_tmd,
                 "dist_to_nearest_TMD": prox,
                 "delta_membrane_insertion": dmi})

feat_tmd = pd.DataFrame(rows)
print(f"计算完成: {len(feat_tmd)} 个变体")
print(f"wt_mismatch (canonical 与 WT 不符): {len(wt_mismatch_log)}")
if wt_mismatch_log:
    for item in wt_mismatch_log[:5]:
        print(f"  KEY={item[0]}, acc={item[1]}, wt={item[2]}, mut={item[3]}, pos={item[4]}")

计算完成: 2179 个变体
wt_mismatch (canonical 与 WT 不符): 0


## 14d. 质检

In [5]:
print(f"\n{'='*60}")
print(f"  质检报告")
print(f"{'='*60}")

# 缺失率
print(f"\n缺失率:")
print(feat_tmd.isna().mean().to_string())

# 值分布
print(f"\nin_TMD 分布:")
print(feat_tmd["in_TMD"].value_counts().to_string())
print(f"  → 正类占比: {feat_tmd['in_TMD'].mean()*100:.1f}%")

print(f"\ndist_to_nearest_TMD 分布:")
print(feat_tmd["dist_to_nearest_TMD"].describe().to_string())

print(f"\ndelta_membrane_insertion 分布:")
dmi_finite = feat_tmd["delta_membrane_insertion"]
print(dmi_finite.describe().to_string())
print(f"  非零占比: {(dmi_finite != 0).mean()*100:.1f}%")

# 与 delta_hydrophobicity 的冗余检查
df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")
# KEY 对齐: features_v3 用 Gene||Variant, TMD 用 Gene||Mutation_used
# 直接用 df_src 的 delta_hydrophobicity (行序一致)
dh = df_src["delta_hydrophobicity"].values
dmi = feat_tmd["delta_membrane_insertion"].values
mask = np.isfinite(dh) & np.isfinite(dmi)
corr = np.corrcoef(dh[mask], dmi[mask])[0, 1]
print(f"\ncorr(delta_hydrophobicity, delta_membrane_insertion) = {corr:.4f}")
if abs(corr) > 0.95:
    print(f"  ⚠ 高度冗余! 建议把 delta_membrane_insertion 换成 is_membrane_protein")
elif abs(corr) > 0.7:
    print(f"  ⚠ 中等相关，注意观察特征重要性是否双双掉队")
else:
    print(f"  ✓ 相关性较低，两个特征捕捉不同信号")

# 按标签分组看 in_TMD
y_bin = (df_feat["reloc_v3"].values > 0).astype(int)
print(f"\n按 reloc 分组:")
print(f"  负例 (不重定位): in_TMD 占比 = {feat_tmd.loc[y_bin==0, 'in_TMD'].mean():.4f}")
print(f"  正例 (重定位):   in_TMD 占比 = {feat_tmd.loc[y_bin==1, 'in_TMD'].mean():.4f}")

print(f"{'='*60}")


  质检报告

缺失率:
KEY                         0.0
in_TMD                      0.0
dist_to_nearest_TMD         0.0
delta_membrane_insertion    0.0

in_TMD 分布:
in_TMD
0    2028
1     151
  → 正类占比: 6.9%

dist_to_nearest_TMD 分布:
count    2179.000000
mean        0.084444
std         0.257522
min         0.000000
25%         0.000000
50%         0.000000
75%         0.004751
max         1.000000

delta_membrane_insertion 分布:
count    2179.000000
mean        0.001886
std         0.023227
min        -0.296983
25%         0.000000
50%         0.000000
75%         0.000000
max         0.226815
  非零占比: 6.9%

corr(delta_hydrophobicity, delta_membrane_insertion) = -0.2146
  ✓ 相关性较低，两个特征捕捉不同信号

按 reloc 分组:
  负例 (不重定位): in_TMD 占比 = 0.0576
  正例 (重定位):   in_TMD 占比 = 0.1660


## 14e. 缺失填充 + 保存

In [6]:
# 缺失填充策略
# in_TMD → 0 (保守: 假设不在 TMD 内)
# dist_to_nearest_TMD → 0 (等价 "无限远")
# delta_membrane_insertion → 0 (保守: 假设无影响)
fill_vals = {"in_TMD": 0, "dist_to_nearest_TMD": 0.0, "delta_membrane_insertion": 0.0}
feat_tmd = feat_tmd.fillna(fill_vals)

out_path = BASE_PATH + "tmd_features.csv"
feat_tmd.to_csv(out_path, index=False, float_format="%.6g")
print(f"已保存: {out_path}")
print(f"形状: {feat_tmd.shape}")
print(f"列: {feat_tmd.columns.tolist()}")
print(f"\n样本:")
print(feat_tmd.head(10).to_string())

已保存: /mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/tmd_features.csv
形状: (2179, 4)
列: ['KEY', 'in_TMD', 'dist_to_nearest_TMD', 'delta_membrane_insertion']

样本:
            KEY  in_TMD  dist_to_nearest_TMD  delta_membrane_insertion
0   LDHA||K222E       0             0.000000                       0.0
1     PTEN||K6N       0             0.000000                       0.0
2   ASS1||K310Q       0             0.000000                       0.0
3   EPHX2||K55R       0             0.000000                       0.0
4    FTH1||K54R       0             0.000000                       0.0
5   GNPTAB||K4Q       0             0.052632                       0.0
6   HRAS||K117R       0             0.000000                       0.0
7   OPTN||K435R       0             0.000000                       0.0
8   IRF4||K123R       0             0.000000                       0.0
9  MAP2K1||K57T       0             0.000000                       0.0


## 14f. (可选) 生成含 TMD 的 64 维特征矩阵

将 tmd_features 拼接到特征矩阵中，供下游 task 直接使用。
下游 task 在加载 ddg 的同时也加载 tmd_features.csv 即可。

**下游加载示例**:
```python
df_tmd = pd.read_csv(BASE_PATH + "tmd_features.csv")
tmd_map = {k: (v1, v2, v3) for k, v1, v2, v3 in 
           zip(df_tmd["KEY"], df_tmd["in_TMD"], 
               df_tmd["dist_to_nearest_TMD"], df_tmd["delta_membrane_insertion"])}
# 按 full_keys (Gene||Mutation_used) 对齐
tmd_arr = np.array([tmd_map.get(k, (0, 0.0, 0.0)) for k in full_keys], dtype=np.float32)
# tmd_arr.shape = (n, 3)，拼到 X_tr / X_te 中
```

## 14g. 下一步

- 在既有 5-fold StratifiedGroupKFold(groups=Gene) 协议下，用 PCA(50) + 7struct + chosen_3 + 3TMD = **63 维**重跑 XGBoost
- 看这 3 个新特征的 **importance** 和 **ablation ΔAUROC**
- 若 `delta_hydrophobicity` 与 `delta_membrane_insertion` 的 corr > 0.95，把后者换成 `is_membrane_protein`